# 06 · Results tables, plots and error analysis

**Runtime:** CPU. **Run after:** any of 02–05 (it summarises whatever has finished). It writes `results_summary.md` and `results_all.csv` to Drive.

In [ ]:
# Setup: Drive, code, packages, paths  (expected runtime: CPU)
from google.colab import drive
drive.mount('/content/drive')
import os, sys
BASE = '/content/drive/MyDrive/neural-evidence'
CODE = '/content/drive/MyDrive/Colab Notebooks/neural-evidence'   # your uploaded code folder
REPO_URL = ''
if not os.path.exists(f'{CODE}/ne') and REPO_URL:
    os.system(f'git clone -q {REPO_URL} /content/code'); CODE = '/content/code'
assert os.path.exists(f'{CODE}/ne'), f'Code not found at {CODE}'
os.chdir(CODE); sys.path.insert(0, CODE)
!pip -q install -r requirements.txt 
PROC, RESULTS, CACHE = f'{BASE}/data/processed', f'{BASE}/results', f'{BASE}/cache'
os.makedirs(RESULTS, exist_ok=True); os.makedirs(CACHE, exist_ok=True)
from ne import runtime
from ne.data import read_jsonl, write_jsonl, eval_sets
runtime.setup()
if runtime.gpu_kind() not in 'CPU'.split('/') and 'CPU' != 'CPU':
    print('WARNING: expected CPU; this runtime is', runtime.gpu_kind(), '- it will still work, just at a different speed/cost')
EVALS = eval_sets(PROC); print('eval sets:', list(EVALS))

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from ne import results, metrics
df = results.collect(RESULTS); S = results.summarize(df)
df.to_csv(f'{BASE}/results_all.csv', index=False)
fmt = lambda m, s: f'{m:.3f} ± {s:.3f}' if s else f'{m:.3f}'
def table(rows, cols=('fever_test', 'fever_test_clean', 'scifact_test')):
    out = []
    for (exp, run), g in rows.groupby(['exp', 'run'], sort=False):
        rec = {'exp': exp, 'run': run, 'seeds': int(g['seeds'].max())}
        for c in cols:
            x = g[g.eval_set == c]
            if len(x):
                x = x.iloc[0]
                rec[f'{c} macro-F1'] = fmt(x.macro_f1, x.macro_f1_std)
                rec[f'{c} R-recall'] = f'{x.R_recall:.3f}'
        rec['train min'] = g['train_min'].max(); rec['peak VRAM GB'] = g['peak_vram_gb'].max()
        out.append(rec)
    return pd.DataFrame(out)
report = []
def show(title, t):
    print('\n## ' + title); display(t); report.append(f'## {title}\n\n' + t.to_markdown(index=False))
pick = lambda pairs: S[S.apply(lambda r: (r.exp, r.run) in pairs, axis=1)]
show('E1 · Baselines vs fine-tuned', table(pick({('E1','majority'),('E1','tfidf_lr'),('E1','bilstm'),('E1','zeroshot_mnli'),('MAIN','deberta_full'),('E2','deberta_v3_base')})))
show('E2 · Architectures (50K subset)', table(S[S.exp == 'E2']))
show('E3 · Loss / imbalance (50K)', table(pick({('E2','deberta_v3_base'),('E3','deberta_weighted_ce'),('E3','deberta_focal')})))
show('E4 · Evidence ablation', table(pick({('E2','deberta_v3_base'),('E4','deberta_claim_only'),('E4','deberta_retrieved'),
                                            ('E1','tfidf_lr'),('E4','tfidf_lr_claim_only'),('E4','tfidf_lr_retrieved'),
                                            ('E1','bilstm'),('E4','bilstm_claim_only'),('E1','zeroshot_mnli_retrieved')})))
show('E5 · SciFact transfer (SciFact dev)', table(S[S.exp == 'E5'], cols=('scifact_dev',)))
show('E6/E7 · Robustness', table(pick({('MAIN','deberta_full'),('E7','deberta_full_para_aug')}),
                                 cols=('fever_test', 'fever_test_distract', 'fever_test_para')))

In [ ]:
# E5 learning curve
e5 = S[(S.exp == 'E5') & S.run.str.startswith('fever_then')].copy()
if len(e5):
    e5['pct'] = e5.run.str.extract(r'(\d+)pct').astype(int)
    e5 = e5.sort_values('pct')
    plt.errorbar(e5.pct, e5.macro_f1, yerr=e5.macro_f1_std, marker='o', label='FEVER → SciFact')
    for run, ls in [('zeroshot_from_fever', '--'), ('scifact_only_100pct', ':')]:
        x = S[(S.exp == 'E5') & (S.run == run)]
        if len(x): plt.axhline(x.macro_f1.iloc[0], ls=ls, color='gray', label=run)
    plt.xlabel('% of SciFact train used'); plt.ylabel('macro-F1 (SciFact dev)'); plt.legend(); plt.grid(alpha=.3)
    plt.savefig(f'{BASE}/e5_learning_curve.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Error analysis on FEVER test for the main model (seed 42)
import re
d = f'{RESULTS}/MAIN/deberta_full_seed42'
if os.path.exists(f'{d}/preds_fever_test.npy'):
    rows = read_jsonl(EVALS['fever_test'])
    pred = np.load(f'{d}/preds_fever_test.npy').argmax(-1)
    y = np.array([r['label'] for r in rows])
    neg = re.compile(r"\b(not|no|never|none|nobody|nothing|neither|nor|without|n't|only|incapable|unable|fails?)\b", re.I)
    cats = {
        'negation in claim': np.array([bool(neg.search(r['claim'])) for r in rows]),
        'number in claim': np.array([bool(re.search(r'\d', r['claim'])) for r in rows]),
        'NEI (retrieved evidence)': y == 2,
        'multi-sentence gold evidence': np.array([len(r.get('gold_ids', [])) > 1 for r in rows]),
        'all': np.ones(len(rows), bool),
    }
    ea = pd.DataFrame([{'category': k, 'n': int(m.sum()), 'error rate': round(float((pred[m] != y[m]).mean()), 3),
                        'macro-F1': round(metrics.evaluate(y[m], pred[m])['macro_f1'], 3)} for k, m in cats.items() if m.sum()])
    show('Error analysis (MAIN seed 42, FEVER test)', ea)
    cm = metrics.evaluate(y, pred)['confusion_matrix']
    show('Confusion matrix (rows = true S/R/NEI)', pd.DataFrame(cm, index=['S','R','NEI'], columns=['S','R','NEI']).reset_index())
    errs = [i for i in range(len(rows)) if pred[i] != y[i]]
    ex = pd.DataFrame([{'true': rows[i]['label_name'], 'pred': metrics.SHORT[['SUPPORTS','REFUTES','NOT ENOUGH INFO'][pred[i]]],
                        'claim': rows[i]['claim'], 'evidence': rows[i]['evidence_gold'][:250]}
                       for i in np.random.default_rng(0).choice(errs, min(25, len(errs)), replace=False)])
    ex.to_csv(f'{BASE}/error_samples.csv', index=False); display(ex)
open(f'{BASE}/results_summary.md', 'w').write('\n\n'.join(report))
print('saved results_summary.md, results_all.csv, error_samples.csv to', BASE)